# Reconstructed seventh-place baseline
Reference MAP@200: **0.7044927571**. Random forest and L1 logistic regression use the original feature masks and weights 0.69 and 0.57.

Run sections 1–3 to retrain five folds using packaged inputs. Section 4 rebuilds preprocessing and features from raw data.

Source: https://github.com/andreiolariu/kaggle-event-recommendation

## 1. Setup

In [ ]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'environment/shared.ipynb').exists())
for cell in json.loads((ROOT / 'environment/shared.ipynb').read_text())['cells']:
    if cell['cell_type'] == 'code':
        exec(''.join(cell['source']), globals())
train = pd.read_csv(ROOT / 'data/raw/train.csv', dtype={'user':str,'event':str})
test = pd.read_csv(ROOT / 'data/raw/test.csv', dtype={'user':str,'event':str})
samples = pd.concat([train.drop(columns=['interested','not_interested']),test],ignore_index=True)
folds = pd.read_csv(ROOT / 'data/folds.csv').fold.to_numpy()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
HERE = ROOT / '01_reconstructed_baseline'
OUT = HERE / 'results'


## 2. Original model and metric
The author metric returns one for users with no positives; the fixed training users all have positives. Feature masks, deduplication and prediction tie order match the historical reproduction.

In [ ]:
class Model:
    def __init__(self, compress=None, has_none=None, C=0.03, n_est=300):
        self.models = [
            RandomForestClassifier(n_estimators=n_est, random_state=42, n_jobs=4, max_features='sqrt'),  # Keep 300 trees and the historical square-root feature sampling.
            LogisticRegression(C=C, penalty='l1', l1_ratio=1, solver='liblinear', random_state=42)  # Preserve L1 regularization and the liblinear solver.
        ]
        self.compress = compress
        self.has_none = has_none

    def fit(self, X, Y):
        X2 = np.array(X)
        if self.compress:
            X3 = X2.compress(self.compress, axis=1)
        self.models[0].fit(X3, Y)
        if self.has_none:
            X3 = X2.compress(self.has_none, axis=1)
        self.models[1].fit(X3, Y)

    def test(self, X):
        X2 = np.array(X)
        if self.compress:
            X3 = X2.compress(self.compress, axis=1)
        rez = self.models[0].predict_proba(X3)[:,1] * 0.69

        if self.has_none:
            X3 = X2.compress(self.has_none, axis=1)
        rez += self.models[1].predict_proba(X3)[:,1] * 0.57

        return rez

def apk(actual, predicted, k=200):
    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)

    if not actual:
        return 1.0

    return score / min(len(actual), k)


## 3. Train and score five folds

In [ ]:
splits = pd.read_pickle(HERE / 'reference/five_fold_features_utc.pkl')
records = []
for fold in range(5):
    held = splits[fold]
    fit = [sum([part[k] for j,part in enumerate(splits) if j != fold], []) for k in range(3)]
    forest_mask, linear_mask = [True]*len(fit[0][0]), [True]*len(fit[0][0])
    for i in [19,20,34,8,11,22,24,28,33,30,31,32]:
        forest_mask[i] = False
    for i in [19,20,21,22,23,24,25,26,29,30,31,32,34,3,4,9,11,14,15,16,17,27,28,30,31,32]:
        linear_mask[i] = False
    model = Model(compress=forest_mask,has_none=linear_mask)
    model.fit(fit[0],fit[1])
    pd.to_pickle(model.models, HERE / 'results' / f'fold_{fold+1}_estimators.pkl')
    predictions = model.test(held[0])
    grouped = {}
    for (user,event),score in zip(held[4],predictions):
        grouped.setdefault(user,[]).append((event,score))
    for user,items in grouped.items():
        items.sort(key=lambda item:(-item[1],str(item[0])))
        records.append(dict(user=user,fold=fold+1,ap200=apk(held[3][user],[item[0] for item in items])))
    print('Fold',fold+1,pd.DataFrame(records).query('fold == @fold + 1').ap200.mean())
result = pd.DataFrame(records)
result.to_csv(HERE / 'results/user_ap.csv',index=False)
print('MAP@200:',result.ap200.mean())


## 4. Rebuild historical inputs from raw data
This reruns full-table clustering and location inference. Results remain separate from the packaged reference cache.

In [ ]:
for name in ['preprocessing.ipynb','features.ipynb']:
    for cell in json.loads((HERE / name).read_text())['cells']:
        if cell['cell_type'] == 'code':
            exec(''.join(cell['source']), globals())
user_info, event_info, attendance, friends_db = load_collections()
rebuilt = get_crossval_data()
print('Rebuilt folds:',len(rebuilt))


## Verified result

Five folds retrained from packaged historical features.

MAP@200: **0.7044927571**, matching the original experiment.
